# GED with all detectors and real preprocessing

In [1]:
import json
import os
import sys

# NOTE: import from root of the project
sys.path.append(os.path.abspath("../../../../"))


from src.services.ged.features.subsystems.lexicon import LexiconDetector
from src.services.ged.features.subsystems.ml import MLDetector
from src.services.ged.features.subsystems.rule_based import RuleBasedDetector
from src.services.ged.fusion import resolve_overlaps
from src.services.ged.orchestrator import GEDService
from src.services.ged.schemas import GEDInput
from src.services.preprocessing.orchestrator import preprocess
from src.services.preprocessing.schemas import PreprocessingInput

2026-06-23 21:24:59.970 | DEBUG    | src.services.ged.features.subsystems.rule_based.registry:register_entry:99 - Registered entry: OT_ALIF_MAQSURA_ALA
2026-06-23 21:24:59.971 | DEBUG    | src.services.ged.features.subsystems.rule_based.registry:register_entry:99 - Registered entry: OT_ALIF_MAQSURA_HATTA
2026-06-23 21:24:59.971 | DEBUG    | src.services.ged.features.subsystems.rule_based.registry:register_entry:99 - Registered entry: OT_TANWIN_NASB_ON_ALIF
2026-06-23 21:24:59.972 | DEBUG    | src.services.ged.features.subsystems.rule_based.registry:register_entry:99 - Registered entry: OT_IDGHAM_AN_MA
2026-06-23 21:24:59.973 | DEBUG    | src.services.ged.features.subsystems.rule_based.registry:register_entry:99 - Registered entry: OT_IDGHAM_MIN_MA
2026-06-23 21:24:59.975 | DEBUG    | src.services.ged.features.subsystems.rule_based.registry:register_entry:99 - Registered entry: OT_TA_MARBUTA_NOUN
2026-06-23 21:24:59.976 | DEBUG    | src.services.ged.features.subsystems.rule_based.regist

In [2]:
rule_detector = RuleBasedDetector()
lexicon_detector = LexiconDetector()
ml_detector = MLDetector()

detectors = [rule_detector, lexicon_detector, ml_detector]

service = GEDService(subsystems=detectors)

print("Loaded detectors:")

for detector in detectors:
    if hasattr(detector, "threshold"):
        details = f" (threshold: {detector.threshold})"
    else:
        details = ""
    print(f"- {detector.name}{details}")

print(f"Loaded {len(rule_detector.list_rules())} rule-based rules")
print(f"Loaded {len(lexicon_detector.list_patterns())} curated lexicon patterns")

2026-06-23 21:25:00.160 | INFO     | src.services.ged.features.subsystems.lexicon.loader:load_patterns:49 - Loaded 2 lexicon patterns from /home/amir/dev/baligh/src/services/ged/features/subsystems/lexicon/resources/patterns.yaml.


Loaded detectors:
- rule_based
- lexicon_matcher
- sequence_labeler (threshold: 0.35)
Loaded 85 rule-based rules
Loaded 2 curated lexicon patterns


In [3]:
CATEGORY_NAMES = {
    "OT": "إملاء",
    "MO": "صرف",
    "SY": "نحو",
    "SE": "دلالة/استعمال",
    "PC": "ترقيم",
    "MG": "دمج يحتاج إلى فصل",
    "SP": "فصل يحتاج إلى دمج",
}


def prepare(text: str):
    """Run preprocessing adapt to GEDInput."""
    pre_output = preprocess(PreprocessingInput(text=text))
    payload = GEDInput(
        text=pre_output.text,
        normalized_text=pre_output.normalized_text,
        tokens=pre_output.tokens,
        morph_features=pre_output.morph_features,
    )
    return pre_output, payload


def print_spans(text: str, spans) -> None:
    """Print spans."""
    if not spans:
        print("  No errors found")
        return
    for error in spans:
        surface = text[error.span[0] : error.span[1]]
        sources = ", ".join(source.value for source in error.sources)
        category = error.category.value
        print(
            f"  - {surface!r}: {category} ({CATEGORY_NAMES[category]}) / "
            f"{error.subtype}; confidence={error.confidence:.3f}; "
            f"span={error.span}; sources=[{sources}]"
        )
        if error.explanation_text:
            print(f"    الشرح: {error.explanation_text}")


def test_all(
    text: str,
    *,
    show_preprocessing: bool = False,
    raw: bool = False,
):
    """Run all detectors."""
    # NOTE: The final pipeline should use the public service interface below.
    # I just to test each
    #
    # pre_output = preprocess(PreprocessingInput(text=text))
    # ged_output = service.process(
    #     GEDInput(
    #         text=pre_output.text,
    #         normalized_text=pre_output.normalized_text,
    #         tokens=pre_output.tokens,
    #         morph_features=pre_output.morph_features,
    #     )
    # )
    # errors = ged_output.errors
    pre_output, payload = prepare(text)
    spans_by_detector = {}

    print(f"\nTEXT: {text}")
    print("TOKENS:", [token.form for token in payload.tokens])

    for detector in detectors:
        spans = detector.detect(
            payload.text,
            payload.normalized_text,
            payload.tokens,
            payload.morph_features,
        )
        spans_by_detector[detector.name] = spans

        print(f"\n[{detector.name}] {len(spans)} error(s)")
        print_spans(payload.text, spans)

    fused = resolve_overlaps(
        [span for spans in spans_by_detector.values() for span in spans]
    )

    print(f"\n[FUSED] {len(fused)} error(s)")
    print_spans(payload.text, fused)

    if show_preprocessing:
        print("\nPREPROCESSING OUTPUT:")
        print(json.dumps(pre_output.model_dump(), ensure_ascii=False, indent=2))
    if raw:
        print("\nRAW FUSED OUTPUT:")
        output = service.process(payload)
        print(json.dumps(output.model_dump(mode="json"), ensure_ascii=False, indent=2))

## Detector data

In [4]:
print("Rule-based rules:")
for rule in rule_detector.list_rules():
    print(f"- {rule.rule_id}: {rule.category.value} / {rule.subtype}")

Rule-based rules:
- OT_ALIF_MAQSURA_ALA: OT / alif_maqsura
- OT_ALIF_MAQSURA_HATTA: OT / alif_maqsura
- OT_TANWIN_NASB_ON_ALIF: OT / tanwin
- OT_IDGHAM_AN_MA: OT / idgham
- OT_IDGHAM_MIN_MA: OT / idgham
- OT_TA_MARBUTA_NOUN: OT / ta_marbuta
- OT_TA_MARBUTA_ADJ: OT / ta_marbuta
- OT_TA_MARBUTA_NOUN_PROP: OT / ta_marbuta
- SE_DECADES_IYAT: SE / lexical_usage
- SE_MOAKHARAN: SE / lexical_usage
- SE_MUTAAKID: SE / lexical_usage
- SE_DHATA: SE / lexical_usage
- SE_KHATIR: SE / lexical_usage
- SE_KHAMMARA: SE / lexical_usage
- SE_INDHAHALA: SE / lexical_usage
- SE_BIAKMALIHI: SE / lexical_usage
- SE_TAHAMMAMA: SE / lexical_usage
- SE_TASHKILU: SE / lexical_usage
- SE_TASAMAMA: SE / lexical_usage
- SE_TATMIN: SE / lexical_usage
- SE_TA3KISU: SE / lexical_usage
- SE_JANOOBI: SE / lexical_usage
- SE_KHESISAN: SE / lexical_usage
- SE_KHALOOQ: SE / lexical_usage
- SE_RAGHMA: SE / lexical_usage
- SE_RAFAH: SE / lexical_usage
- SE_SHAWYAN: SE / lexical_usage
- SE_ARAYA: SE / lexical_usage
- SE_LIWA

In [5]:
print("Lexicon patterns:")
for pattern in lexicon_detector.list_patterns():
    print(f"- {pattern.id}: {pattern.category.value} / {pattern.subtype}")

Lexicon patterns:
- LEX_SPLIT_LAKIN: SP / common_split
- LEX_MERGE_IN_SHA_ALLAH: MG / common_merge


In [6]:
print("ML labels:", ml_detector.error_labels)

ML labels: ('OT', 'PC', 'SY', 'SP', 'UNK', 'MG', 'MO')


## Orthography and punctuation

In [7]:
# ذهبت إلى المدرسة.
test_all("ذهبت الى المدرسة.")

[2026-06-23 21:25:00,914 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.



TEXT: ذهبت الى المدرسة.
TOKENS: ['ذهبت', 'الى', 'المدرسة', '.']

[rule_based] 1 error(s)
  - 'الى': OT (إملاء) / hamza; confidence=1.000; span=(5, 8); sources=[rule_based]
    الشرح: حروف الجر والربط وبعض الأدوات التي أصلها بهمزة قطع تكتب بالهمزة لا بالألف المجردة، مثل: إلى، أو، إذا، أن، إن.

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 1 error(s)
  - 'الى': OT (إملاء) / ml_orthography; confidence=0.987; span=(5, 8); sources=[sequence_labeler]

[FUSED] 1 error(s)
  - 'الى': OT (إملاء) / hamza; confidence=1.000; span=(5, 8); sources=[rule_based, sequence_labeler]
    الشرح: حروف الجر والربط وبعض الأدوات التي أصلها بهمزة قطع تكتب بالهمزة لا بالألف المجردة، مثل: إلى، أو، إذا، أن، إن.


In [8]:
# انتظرت حتى المساء.
test_all("انتظرت حتي المساء.")


TEXT: انتظرت حتي المساء.
TOKENS: ['انتظرت', 'حتي', 'المساء', '.']

[rule_based] 1 error(s)
  - 'حتي': OT (إملاء) / alif_maqsura; confidence=1.000; span=(7, 10); sources=[rule_based]
    الشرح: الصواب «حتى» لا «حتي».

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 1 error(s)
  - 'حتي': OT (إملاء) / ml_orthography; confidence=0.968; span=(7, 10); sources=[sequence_labeler]

[FUSED] 1 error(s)
  - 'حتي': OT (إملاء) / alif_maqsura; confidence=1.000; span=(7, 10); sources=[rule_based, sequence_labeler]
    الشرح: الصواب «حتى» لا «حتي».


In [9]:
# هذه مدرسة كبيرة.
test_all("هذه مدرسه كبيرة.")


TEXT: هذه مدرسه كبيرة.
TOKENS: ['هذه', 'مدرسه', 'كبيرة', '.']

[rule_based] 1 error(s)
  - 'مدرسه': OT (إملاء) / ta_marbuta; confidence=1.000; span=(4, 9); sources=[rule_based]
    الشرح: الاسم المؤنث يُكتب بتاء مربوطة (ة) لا هاء (ه).

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 1 error(s)
  - 'مدرسه': OT (إملاء) / ml_orthography; confidence=0.898; span=(4, 9); sources=[sequence_labeler]

[FUSED] 1 error(s)
  - 'مدرسه': OT (إملاء) / ta_marbuta; confidence=1.000; span=(4, 9); sources=[rule_based, sequence_labeler]
    الشرح: الاسم المؤنث يُكتب بتاء مربوطة (ة) لا هاء (ه).


In [10]:
# أعرف أنه مجتهد
test_all("أعرف انه مجتهد.")


TEXT: أعرف انه مجتهد.
TOKENS: ['أعرف', 'انه', 'مجتهد', '.']

[rule_based] 1 error(s)
  - 'انه': OT (إملاء) / hamza; confidence=1.000; span=(5, 8); sources=[rule_based]
    الشرح: حروف الجر والربط وبعض الأدوات التي أصلها بهمزة قطع تكتب بالهمزة لا بالألف المجردة، مثل: إلى، أو، إذا، أن، إن.

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 1 error(s)
  - 'انه': OT (إملاء) / ml_orthography; confidence=0.987; span=(5, 8); sources=[sequence_labeler]

[FUSED] 1 error(s)
  - 'انه': OT (إملاء) / hamza; confidence=1.000; span=(5, 8); sources=[rule_based, sequence_labeler]
    الشرح: حروف الجر والربط وبعض الأدوات التي أصلها بهمزة قطع تكتب بالهمزة لا بالألف المجردة، مثل: إلى، أو، إذا، أن، إن.


In [11]:
# هذه حجة قوية.
test_all("هذه حجة قويه.")


TEXT: هذه حجة قويه.
TOKENS: ['هذه', 'حجة', 'قويه', '.']

[rule_based] 0 error(s)
  No errors found

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 1 error(s)
  - 'قويه': OT (إملاء) / ml_orthography; confidence=0.999; span=(8, 12); sources=[sequence_labeler]

[FUSED] 1 error(s)
  - 'قويه': OT (إملاء) / ml_orthography; confidence=0.999; span=(8, 12); sources=[sequence_labeler]


In [12]:
# وصلت فاطمة مبكرًا.
test_all("وصلت فاطمه مبكرا.", show_preprocessing=True)


TEXT: وصلت فاطمه مبكرا.
TOKENS: ['وصلت', 'فاطمه', 'مبكرا', '.']

[rule_based] 1 error(s)
  - 'فاطمه': OT (إملاء) / ta_marbuta; confidence=1.000; span=(5, 10); sources=[rule_based]
    الشرح: الاسم العلم المؤنث يُكتب بتاء مربوطة (ة) لا هاء (ه)، مثل: مكة، فاطمة.

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 1 error(s)
  - 'فاطمه': OT (إملاء) / ml_orthography; confidence=0.991; span=(5, 10); sources=[sequence_labeler]

[FUSED] 1 error(s)
  - 'فاطمه': OT (إملاء) / ta_marbuta; confidence=1.000; span=(5, 10); sources=[rule_based, sequence_labeler]
    الشرح: الاسم العلم المؤنث يُكتب بتاء مربوطة (ة) لا هاء (ه)، مثل: مكة، فاطمة.

PREPROCESSING OUTPUT:
{
  "text": "وصلت فاطمه مبكرا.",
  "normalized_text": "وصلت فاطمه مبكرا.",
  "tokens": [
    {
      "index": 0,
      "form": "وصلت",
      "span": [
        0,
        4
      ],
      "norm_span": [
        0,
        4
      ],
      "affix_structure": "STEM+PRON",
      "farasa_segmentation": "وصل+ت",
      "is_oov": f

In [13]:
# ذهب خالد، ثم عاد.
test_all("ذهب خالد ، ثم عاد.")


TEXT: ذهب خالد ، ثم عاد.
TOKENS: ['ذهب', 'خالد', '،', 'ثم', 'عاد', '.']

[rule_based] 1 error(s)
  - '،': PC (ترقيم) / spacing; confidence=1.000; span=(9, 10); sources=[rule_based]
    الشرح: علامة الترقيم يجب أن تلتصق بالكلمة التي تسبقها دون فراغ؛ مثل: «ذهب، ثم» لا «ذهب ، ثم»

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 1 error(s)
  - '،': PC (ترقيم) / spacing; confidence=1.000; span=(9, 10); sources=[rule_based]
    الشرح: علامة الترقيم يجب أن تلتصق بالكلمة التي تسبقها دون فراغ؛ مثل: «ذهب، ثم» لا «ذهب ، ثم»


## Syntax and agreement

In [14]:
# ذهب الطلاب إلى الفصل.
test_all("ذهبوا الطلاب إلى الفصل.")


TEXT: ذهبوا الطلاب إلى الفصل.
TOKENS: ['ذهبوا', 'الطلاب', 'إلى', 'الفصل', '.']

[rule_based] 1 error(s)
  - 'ذهبوا': SY (نحو) / verb_subject_agreement; confidence=1.000; span=(0, 5); sources=[rule_based]
    الشرح: إذا تقدَّم الفعل على الفاعل وجب إفراد الفعل وتجريده من علامة التثنية أو الجمع، مثل: «ذهب الطلاب» لا «ذهبوا الطلاب»

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 1 error(s)
  - 'ذهبوا': SY (نحو) / verb_subject_agreement; confidence=1.000; span=(0, 5); sources=[rule_based]
    الشرح: إذا تقدَّم الفعل على الفاعل وجب إفراد الفعل وتجريده من علامة التثنية أو الجمع، مثل: «ذهب الطلاب» لا «ذهبوا الطلاب»


In [15]:
# قرأت الكتاب المفيد.
test_all("قرأت الكتاب مفيد.")


TEXT: قرأت الكتاب مفيد.
TOKENS: ['قرأت', 'الكتاب', 'مفيد', '.']

[rule_based] 1 error(s)
  - 'مفيد': SY (نحو) / noun_adjective_agreement; confidence=1.000; span=(12, 16); sources=[rule_based]
    الشرح: النعت يتبع المنعوت في التعريف والتنكير؛ فإن كان الاسم معرفةً وجب تعريف النعت، وإن كان نكرةً وجب تنكيره

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 1 error(s)
  - 'مفيد': SY (نحو) / noun_adjective_agreement; confidence=1.000; span=(12, 16); sources=[rule_based]
    الشرح: النعت يتبع المنعوت في التعريف والتنكير؛ فإن كان الاسم معرفةً وجب تعريف النعت، وإن كان نكرةً وجب تنكيره


In [16]:
# هذه السلامة مهمة.
test_all("هذا السلامة مهمة.")


TEXT: هذا السلامة مهمة.
TOKENS: ['هذا', 'السلامة', 'مهمة', '.']

[rule_based] 2 error(s)
  - 'مهمة': SY (نحو) / noun_adjective_agreement; confidence=1.000; span=(12, 16); sources=[rule_based]
    الشرح: النعت يتبع المنعوت في التعريف والتنكير؛ فإن كان الاسم معرفةً وجب تعريف النعت، وإن كان نكرةً وجب تنكيره
  - 'هذا': SY (نحو) / demonstrative_noun_gender; confidence=1.000; span=(0, 3); sources=[rule_based]
    الشرح: اسم الإشارة يجب أن يطابق الاسم الذي بعده في التذكير والتأنيث؛ فنقول: «هذه السلامة» و«هذا البطل»

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 2 error(s)
  - 'هذا': SY (نحو) / demonstrative_noun_gender; confidence=1.000; span=(0, 3); sources=[rule_based]
    الشرح: اسم الإشارة يجب أن يطابق الاسم الذي بعده في التذكير والتأنيث؛ فنقول: «هذه السلامة» و«هذا البطل»
  - 'مهمة': SY (نحو) / noun_adjective_agreement; confidence=1.000; span=(12, 16); sources=[rule_based]
    الشرح: النعت يتبع المنعوت في التعريف والتنكير؛ فإن كا

In [17]:
# سمعت القول الذي انتشر.
test_all("سمعت القول التي انتشر.")


TEXT: سمعت القول التي انتشر.
TOKENS: ['سمعت', 'القول', 'التي', 'انتشر', '.']

[rule_based] 1 error(s)
  - 'التي': SY (نحو) / relative_pronoun_gender; confidence=1.000; span=(11, 15); sources=[rule_based]
    الشرح: الاسم الموصول يجب أن يطابق الاسم السابق له في التذكير والتأنيث؛ فنقول: «القول الذي» و«الوشاية التي»

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 1 error(s)
  - 'التي': SY (نحو) / relative_pronoun_gender; confidence=1.000; span=(11, 15); sources=[rule_based]
    الشرح: الاسم الموصول يجب أن يطابق الاسم السابق له في التذكير والتأنيث؛ فنقول: «القول الذي» و«الوشاية التي»


In [18]:
# بحثت في الكتابين.
test_all("بحثت في الكتابان.")


TEXT: بحثت في الكتابان.
TOKENS: ['بحثت', 'في', 'الكتابان', '.']

[rule_based] 1 error(s)
  - 'الكتابان': SY (نحو) / preposition_dual_case; confidence=1.000; span=(8, 16); sources=[rule_based]
    الشرح: الاسم المثنى بعد حرف الجر يجب أن يكون مجرورًا؛ فنقول: «في الكتابين» لا «في الكتابان»

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 1 error(s)
  - 'الكتابان': SY (نحو) / preposition_dual_case; confidence=1.000; span=(8, 16); sources=[rule_based]
    الشرح: الاسم المثنى بعد حرف الجر يجب أن يكون مجرورًا؛ فنقول: «في الكتابين» لا «في الكتابان»


In [19]:
# سافرت مع المسافرين.
test_all("سافرت مع المسافرون.")


TEXT: سافرت مع المسافرون.
TOKENS: ['سافرت', 'مع', 'المسافرون', '.']

[rule_based] 1 error(s)
  - 'المسافرون': SY (نحو) / preposition_sound_masc_plural_case; confidence=1.000; span=(9, 18); sources=[rule_based]
    الشرح: جمع المذكر السالم بعد حرف الجر يجب أن يكون مجرورًا؛ فنقول: «مع المسافرين» لا «مع المسافرون»

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 1 error(s)
  - 'المسافرون': SY (نحو) / preposition_sound_masc_plural_case; confidence=1.000; span=(9, 18); sources=[rule_based]
    الشرح: جمع المذكر السالم بعد حرف الجر يجب أن يكون مجرورًا؛ فنقول: «مع المسافرين» لا «مع المسافرون»


## Lexicon: split, merge, and spelling

In [20]:
# إن شاء الله يكون الأمر خيرًا.
test_all("انشاءالله يكون الأمر خيرًا.")


TEXT: انشاءالله يكون الأمر خيرًا.
TOKENS: ['انشاءالله', 'يكون', 'الأمر', 'خيرًا', '.']

[rule_based] 0 error(s)
  No errors found

[lexicon_matcher] 1 error(s)
  - 'انشاءالله': MG (دمج يحتاج إلى فصل) / common_merge; confidence=1.000; span=(0, 9); sources=[lexicon_matcher]
    الشرح: الصواب فصل العبارة: إن شاء الله.

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 1 error(s)
  - 'انشاءالله': MG (دمج يحتاج إلى فصل) / common_merge; confidence=1.000; span=(0, 9); sources=[lexicon_matcher]
    الشرح: الصواب فصل العبارة: إن شاء الله.


In [21]:
# ذهبت إلى المدرسة.
test_all("ذهبت إلى المدرثه.")


TEXT: ذهبت إلى المدرثه.
TOKENS: ['ذهبت', 'إلى', 'المدرثه', '.']

[rule_based] 0 error(s)
  No errors found

[lexicon_matcher] 1 error(s)
  - 'المدرثه': OT (إملاء) / spelling; confidence=0.800; span=(9, 16); sources=[lexicon_matcher]
    الشرح: الكلمة غير موجودة في المعجم المتاح ولم يقدم المحلل الصرفي تحليلا موثوقا.

[sequence_labeler] 1 error(s)
  - 'المدرثه': OT (إملاء) / ml_orthography; confidence=0.902; span=(9, 16); sources=[sequence_labeler]

[FUSED] 1 error(s)
  - 'المدرثه': OT (إملاء) / ml_orthography; confidence=0.902; span=(9, 16); sources=[sequence_labeler, lexicon_matcher]


In [22]:
# قابلت أحمد السيد.
test_all("قابلت أحمد السيد.")


TEXT: قابلت أحمد السيد.
TOKENS: ['قابلت', 'أحمد', 'السيد', '.']

[rule_based] 0 error(s)
  No errors found

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 0 error(s)
  No errors found


## Common lexical/semantic usage

In [23]:
# ذهب وحده إلى المنزل.
test_all("ذهب لوحده إلى المنزل.")


TEXT: ذهب لوحده إلى المنزل.
TOKENS: ['ذهب', 'لوحده', 'إلى', 'المنزل', '.']

[rule_based] 2 error(s)
  - 'لوحده': OT (إملاء) / ta_marbuta; confidence=1.000; span=(4, 9); sources=[rule_based]
    الشرح: الاسم المؤنث يُكتب بتاء مربوطة (ة) لا هاء (ه).
  - 'لوحده': SE (دلالة/استعمال) / lexical_usage; confidence=1.000; span=(4, 9); sources=[rule_based]
    الشرح: الأفصح: «وحده» بدل «لوحده»

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 1 error(s)
  - 'لوحده': MO (صرف) / ml_morphology; confidence=0.855; span=(4, 9); sources=[sequence_labeler]

[FUSED] 1 error(s)
  - 'لوحده': OT (إملاء) / ta_marbuta; confidence=1.000; span=(4, 9); sources=[rule_based]
    الشرح: الاسم المؤنث يُكتب بتاء مربوطة (ة) لا هاء (ه).


In [24]:
# نحتاج إلى طمأنة الجمهور.
test_all("نحتاج إلى تطمين الجمهور.")


TEXT: نحتاج إلى تطمين الجمهور.
TOKENS: ['نحتاج', 'إلى', 'تطمين', 'الجمهور', '.']

[rule_based] 1 error(s)
  - 'تطمين': SE (دلالة/استعمال) / lexical_usage; confidence=1.000; span=(10, 15); sources=[rule_based]
    الشرح: الأفصح: «طمأنة» لا «تطمين»

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 1 error(s)
  - 'تطمين': SE (دلالة/استعمال) / lexical_usage; confidence=1.000; span=(10, 15); sources=[rule_based]
    الشرح: الأفصح: «طمأنة» لا «تطمين»


In [25]:
# حدث ذلك في الثلاثينيات.
test_all("حدث ذلك في الثلاثينات.")


TEXT: حدث ذلك في الثلاثينات.
TOKENS: ['حدث', 'ذلك', 'في', 'الثلاثينات', '.']

[rule_based] 1 error(s)
  - 'الثلاثينات': SE (دلالة/استعمال) / lexical_usage; confidence=1.000; span=(11, 21); sources=[rule_based]
    الشرح: يستحسن في العقود أن تكتب بصيغة «ـينيات» لا «ـينات»، مثل: الثلاثينيات

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 1 error(s)
  - 'الثلاثينات': SE (دلالة/استعمال) / lexical_usage; confidence=1.000; span=(11, 21); sources=[rule_based]
    الشرح: يستحسن في العقود أن تكتب بصيغة «ـينيات» لا «ـينات»، مثل: الثلاثينيات


## ML detector and fusion

In [26]:
# هذا الطالب ذهب إلى المدرسة وحده.
test_all("هاذا الطالب ذهب الى المدسه وحده.")


TEXT: هاذا الطالب ذهب الى المدسه وحده.
TOKENS: ['هاذا', 'الطالب', 'ذهب', 'الى', 'المدسه', 'وحده', '.']

[rule_based] 1 error(s)
  - 'الى': OT (إملاء) / hamza; confidence=1.000; span=(16, 19); sources=[rule_based]
    الشرح: حروف الجر والربط وبعض الأدوات التي أصلها بهمزة قطع تكتب بالهمزة لا بالألف المجردة، مثل: إلى، أو، إذا، أن، إن.

[lexicon_matcher] 2 error(s)
  - 'هاذا': OT (إملاء) / spelling; confidence=0.800; span=(0, 4); sources=[lexicon_matcher]
    الشرح: الكلمة غير موجودة في المعجم المتاح ولم يقدم المحلل الصرفي تحليلا موثوقا.
  - 'المدسه': OT (إملاء) / spelling; confidence=0.800; span=(20, 26); sources=[lexicon_matcher]
    الشرح: الكلمة غير موجودة في المعجم المتاح ولم يقدم المحلل الصرفي تحليلا موثوقا.

[sequence_labeler] 3 error(s)
  - 'هاذا': OT (إملاء) / ml_orthography; confidence=0.996; span=(0, 4); sources=[sequence_labeler]
  - 'الى': OT (إملاء) / ml_orthography; confidence=0.969; span=(16, 19); sources=[sequence_labeler]
  - 'المدسه': OT (إملاء) / ml_orthography; confid

In [27]:
# ذهب المعلمون إلى الحديقة.
test_all("ذهبوا المعلمون الي الحدددديقققه.")


TEXT: ذهبوا المعلمون الي الحدددديقققه.
TOKENS: ['ذهبوا', 'المعلمون', 'الي', 'الحدددديقققه', '.']

[rule_based] 2 error(s)
  - 'الي': OT (إملاء) / hamza; confidence=1.000; span=(15, 18); sources=[rule_based]
    الشرح: حروف الجر والربط وبعض الأدوات التي أصلها بهمزة قطع تكتب بالهمزة لا بالألف المجردة، مثل: إلى، أو، إذا، أن، إن.
  - 'ذهبوا': SY (نحو) / verb_subject_agreement; confidence=1.000; span=(0, 5); sources=[rule_based]
    الشرح: إذا تقدَّم الفعل على الفاعل وجب إفراد الفعل وتجريده من علامة التثنية أو الجمع، مثل: «ذهب الطلاب» لا «ذهبوا الطلاب»

[lexicon_matcher] 1 error(s)
  - 'الحدددديقققه': OT (إملاء) / spelling; confidence=0.800; span=(19, 31); sources=[lexicon_matcher]
    الشرح: الكلمة غير موجودة في المعجم المتاح ولم يقدم المحلل الصرفي تحليلا موثوقا.

[sequence_labeler] 1 error(s)
  - 'الي': SY (نحو) / ml_syntax; confidence=0.905; span=(15, 18); sources=[sequence_labeler]

[FUSED] 3 error(s)
  - 'ذهبوا': SY (نحو) / verb_subject_agreement; confidence=1.000; span=(0, 5); source

In [28]:
# إن شاء الله ينجح، لكن عليه أن يدرس.
test_all("انشاءالله ينجح ، لا كن عليه أن يدرس.")


TEXT: انشاءالله ينجح ، لا كن عليه أن يدرس.
TOKENS: ['انشاءالله', 'ينجح', '،', 'لا', 'كن', 'عليه', 'أن', 'يدرس', '.']

[rule_based] 1 error(s)
  - '،': PC (ترقيم) / spacing; confidence=1.000; span=(15, 16); sources=[rule_based]
    الشرح: علامة الترقيم يجب أن تلتصق بالكلمة التي تسبقها دون فراغ؛ مثل: «ذهب، ثم» لا «ذهب ، ثم»

[lexicon_matcher] 2 error(s)
  - 'لا كن': SP (فصل يحتاج إلى دمج) / common_split; confidence=1.000; span=(17, 22); sources=[lexicon_matcher]
    الشرح: تكتب الكلمة متصلة: لكن.
  - 'انشاءالله': MG (دمج يحتاج إلى فصل) / common_merge; confidence=1.000; span=(0, 9); sources=[lexicon_matcher]
    الشرح: الصواب فصل العبارة: إن شاء الله.

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 3 error(s)
  - 'انشاءالله': MG (دمج يحتاج إلى فصل) / common_merge; confidence=1.000; span=(0, 9); sources=[lexicon_matcher]
    الشرح: الصواب فصل العبارة: إن شاء الله.
  - '،': PC (ترقيم) / spacing; confidence=1.000; span=(15, 16); sources=[rule_based]
    الشرح: علامة الترقيم يجب أن

## Clean-sentence negative controls

In [29]:
test_all("ذهبت إلى المدرسة.")


TEXT: ذهبت إلى المدرسة.
TOKENS: ['ذهبت', 'إلى', 'المدرسة', '.']

[rule_based] 0 error(s)
  No errors found

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 0 error(s)
  No errors found


In [30]:
# ذهب الطلاب إلى الفصل.
test_all("ذهب الطلاب إلى الفصل.")


TEXT: ذهب الطلاب إلى الفصل.
TOKENS: ['ذهب', 'الطلاب', 'إلى', 'الفصل', '.']

[rule_based] 0 error(s)
  No errors found

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 0 error(s)
  No errors found


In [33]:
# اكتب الصياغة الصحيحة المرجعية هنا.
test_all("اكتب الخطأ")


TEXT: اكتب الخطأ
TOKENS: ['اكتب']

[rule_based] 0 error(s)
  No errors found

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 1 error(s)
  - 'اكتب': OT (إملاء) / ml_orthography; confidence=0.912; span=(0, 4); sources=[sequence_labeler]

[FUSED] 1 error(s)
  - 'اكتب': OT (إملاء) / ml_orthography; confidence=0.912; span=(0, 4); sources=[sequence_labeler]


---

## Open test


In [32]:
# جرب
text = "بليغ سيميتني بكامل الجدية."
result = test_all(text, show_preprocessing=False, raw=True)


TEXT: بليغ سيميتني بكامل الجدية.
TOKENS: ['بليغ', 'سيميتني', 'بكامل', 'الجدية', '.']

[rule_based] 0 error(s)
  No errors found

[lexicon_matcher] 0 error(s)
  No errors found

[sequence_labeler] 0 error(s)
  No errors found

[FUSED] 0 error(s)
  No errors found

RAW FUSED OUTPUT:
{
  "text": "بليغ سيميتني بكامل الجدية.",
  "errors": []
}
